# Disaster Tweet Classification
## TF-IDF, Linear SVM & Group-Aware Validation

### Project Overview

This notebook presents a reproducible Machine Learning workflow for the Kaggle **Natural Language Processing with Disaster Tweets** competition.

**Objective:** Classify whether a tweet describes a real disaster (`1`) or not (`0`).

### Methodology

1. Exploratory Data Analysis and data quality checks.
2. Group-aware train/validation splitting.
3. TF-IDF vectorization with data leakage prevention.
4. Comparison of Multinomial Naive Bayes, Logistic Regression and Linear SVM.
5. Group-aware cross-validation and model selection.
6. Error analysis and interpretation.
7. Final model training and Kaggle submission generation.

**Selected model:** LinearSVC (C=1.0)

**Previous official Kaggle F1-score:** 0.79313

The notebook documents a reproducible workflow, including methodological limitations and the distinction between validation performance and competition results.

In [1]:
from pathlib import Path

import pandas as pd
import numpy as np
import sklearn

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("Scikit-learn:", sklearn.__version__)

print("\nAvailable input files:")

for path in sorted(Path("/kaggle/input").rglob("*.csv")):
    print(path)

Pandas: 2.3.3
NumPy: 2.1.3
Scikit-learn: 1.6.1

Available input files:
/kaggle/input/competitions/nlp-getting-started/sample_submission.csv
/kaggle/input/competitions/nlp-getting-started/test.csv
/kaggle/input/competitions/nlp-getting-started/train.csv


In [2]:
DATA_DIR = Path("/kaggle/input/competitions/nlp-getting-started")

train_df = pd.read_csv(DATA_DIR / "train.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

print("Train:", train_df.shape)
print("Test:", test_df.shape)
print("Sample submission:", sample_submission.shape)

display(train_df.head())

Train: (7613, 5)
Test: (3263, 4)
Sample submission: (3263, 2)


,id,keyword,location,text,target
0,1,NaN,NaN,Our Deeds are the Reason of this #earthquake M...,1
1,4,NaN,NaN,Forest fire near La Ronge Sask. Canada,1
2,5,NaN,NaN,All residents asked to 'shelter in place' are ...,1
3,6,NaN,NaN,"13,000 people receive #wildfires evacuation or...",1
4,7,NaN,NaN,Just got sent this photo from Ruby #Alaska as ...,1


In [3]:
print("Training columns:")
print(train_df.columns.tolist())

print("\nMissing values:")
print(train_df.isna().sum())

print("\nTarget distribution:")
print(train_df["target"].value_counts())

print("\nDuplicated IDs:")
print(train_df["id"].duplicated().sum())

print("\nMissing tweet texts:")
print(train_df["text"].isna().sum())

Training columns:
['id', 'keyword', 'location', 'text', 'target']

Missing values:
id             0
keyword       61
location    2533
text           0
target         0
dtype: int64

Target distribution:
target
0    4342
1    3271
Name: count, dtype: int64

Duplicated IDs:
0

Missing tweet texts:
0


### Initial Data Quality Findings

The training dataset contains 7,613 tweets, while the official test dataset contains 3,263 unlabeled observations.

**Key findings:**

- 4,342 tweets belong to class 0 (non-disaster).
- 3,271 tweets belong to class 1 (disaster).
- 61 missing values were identified in `keyword`.
- 2,533 missing values were identified in `location`.
- No missing values were found in `text` or `target`.
- No duplicated tweet IDs were found.

The target distribution is moderately imbalanced, so Accuracy alone is not sufficient to evaluate classification performance.

Because the current model uses only tweet text, missing `keyword` and `location` values do not require dropping observations.

Further analysis will investigate repeated texts and conflicting labels, which may affect model evaluation and data splitting.

### Duplicate Texts and Conflicting Labels

Although tweet IDs are unique, different records may contain identical text.

Repeated texts can introduce data leakage if the same message appears in both training and validation partitions.

Additionally, identical texts may have conflicting target labels, indicating potential annotation inconsistencies.

We investigate these issues before splitting the data.

In [4]:
# Exact duplicate texts
duplicate_occurrences = train_df["text"].duplicated().sum()

# Number of distinct text values
unique_texts = train_df["text"].nunique()

# Number of distinct texts associated with conflicting labels
label_counts = train_df.groupby("text")["target"].nunique()
conflicting_texts = (label_counts > 1).sum()

print("Total observations:", len(train_df))
print("Unique texts:", unique_texts)
print("Additional duplicate-text occurrences:", duplicate_occurrences)
print("Distinct texts with conflicting labels:", conflicting_texts)

assert unique_texts + duplicate_occurrences == len(train_df)

Total observations: 7613
Unique texts: 7503
Additional duplicate-text occurrences: 110
Distinct texts with conflicting labels: 18


### Group-Aware Train/Validation Split

We use GroupShuffleSplit to prevent identical tweet texts from appearing in both training and validation sets.

Each unique text acts as a group identifier.

The split uses a fixed random seed for reproducibility. Since GroupShuffleSplit is not stratified, class distributions are checked after splitting.

In [5]:
from sklearn.model_selection import GroupShuffleSplit

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=42
)

train_idx, val_idx = next(
    splitter.split(
        train_df,
        train_df["target"],
        groups=train_df["text"]
    )
)

dev_train = train_df.iloc[train_idx].copy()
dev_val = train_df.iloc[val_idx].copy()

# Ensure identical texts never cross partitions
train_texts = set(dev_train["text"])
val_texts = set(dev_val["text"])

overlap = train_texts.intersection(val_texts)

assert len(overlap) == 0
assert len(dev_train) + len(dev_val) == len(train_df)

print("Training samples:", len(dev_train))
print("Validation samples:", len(dev_val))
print("Overlapping texts:", len(overlap))

print("\nTraining class distribution:")
print(dev_train["target"].value_counts(normalize=True))

print("\nValidation class distribution:")
print(dev_val["target"].value_counts(normalize=True))

Training samples: 6083
Validation samples: 1530
Overlapping texts: 0

Training class distribution:
target
0    0.573401
1    0.426599
Name: proportion, dtype: float64

Validation class distribution:
target
0    0.55817
1    0.44183
Name: proportion, dtype: float64


### Baseline Model Comparison

Three supervised classifiers are evaluated using the same group-aware training and validation partitions:

- Multinomial Naive Bayes
- Logistic Regression
- Linear Support Vector Classifier (LinearSVC)

Each classifier is wrapped in a Scikit-learn Pipeline containing a TF-IDF vectorizer.

The vectorizer is fitted exclusively on the training partition, preventing validation texts from influencing the learned vocabulary and IDF statistics.

Models are compared using Accuracy, Precision, Recall, F1-score, and confusion matrix counts.

These results describe development validation performance, not independent final test performance.

In [6]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

candidates = {
    "MultinomialNB": MultinomialNB(alpha=1.0),

    "LogisticRegression": LogisticRegression(
        C=1.0,
        max_iter=1000,
        random_state=42
    ),

    "LinearSVC": LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    )
}

results = []

for name, classifier in candidates.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", classifier)
    ])

    pipeline.fit(
        dev_train["text"],
        dev_train["target"]
    )

    predictions = pipeline.predict(dev_val["text"])

    tn, fp, fn, tp = confusion_matrix(
        dev_val["target"],
        predictions,
        labels=[0, 1]
    ).ravel()

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(
            dev_val["target"], predictions
        ),
        "Precision": precision_score(
            dev_val["target"], predictions
        ),
        "Recall": recall_score(
            dev_val["target"], predictions
        ),
        "F1": f1_score(
            dev_val["target"], predictions
        ),
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp
    })

results_df = pd.DataFrame(results)

display(
    results_df.sort_values(
        "F1",
        ascending=False
    ).reset_index(drop=True).round(4)
)

,Model,Accuracy,Precision,Recall,F1,TN,FP,FN,TP
0,LinearSVC,0.7948,0.7891,0.7308,0.7588,722,132,182,494
1,LogisticRegression,0.7954,0.8092,0.7027,0.7522,742,112,201,475
2,MultinomialNB,0.7863,0.8643,0.6124,0.7169,789,65,262,414


### Model Comparison — Results and Interpretation

All three classifiers were evaluated on the same group-aware validation partition containing 1,530 tweets.

**LinearSVC achieved the highest observed F1-score (0.7588) and recall (0.7308).**

Logistic Regression achieved slightly higher accuracy and precision than LinearSVC, while Multinomial Naive Bayes achieved the highest precision but substantially lower recall.

LinearSVC identified 19 additional disaster-related tweets compared with Logistic Regression, at the cost of 20 additional false positives.

These findings illustrate the trade-off between precision and recall. The appropriate choice depends on the operational costs of missed disasters and false alarms.

The results come from a single development validation split and do not establish statistical superiority or production readiness.

### Group-Aware Cross-Validation and Model Selection

We compare four classifier configurations using `GroupKFold(n_splits=3)` on the development training partition only.

Identical tweet texts are grouped together to prevent exact-text overlap between training and validation folds.

Each classifier is evaluated through a Pipeline containing `TfidfVectorizer`, ensuring that vocabulary and IDF statistics are learned exclusively from the training portion of each fold.

The original 1,530-observation development validation partition is excluded from this cross-validation experiment.

We report mean F1-score, F1 standard deviation, mean precision, and mean recall.

These scores support candidate selection but should not be interpreted as independent final generalization estimates.

In [7]:
from sklearn.model_selection import GroupKFold, cross_validate

cv_candidates = {
    "NaiveBayes_alpha1": MultinomialNB(alpha=1.0),

    "LogisticRegression_C1": LogisticRegression(
        C=1.0, max_iter=1000, random_state=42
    ),

    "LogisticRegression_C10": LogisticRegression(
        C=10.0, max_iter=1000, random_state=42
    ),

    "LinearSVC_C1": LinearSVC(
        C=1.0, max_iter=5000, random_state=42
    )
}

texts = dev_train["text"]
labels = dev_train["target"]
groups = dev_train["text"]

cv = GroupKFold(n_splits=3)

folds = list(
    cv.split(texts, labels, groups=groups)
)

cv_results = []

for name, classifier in cv_candidates.items():

    pipeline = Pipeline([
        ("tfidf", TfidfVectorizer()),
        ("classifier", classifier)
    ])

    scores = cross_validate(
        pipeline,
        texts,
        labels,
        cv=folds,
        scoring={
            "f1": "f1",
            "precision": "precision",
            "recall": "recall"
        },
        n_jobs=1
    )

    cv_results.append({
        "Model": name,
        "Mean F1": scores["test_f1"].mean(),
        "F1 Std": scores["test_f1"].std(),
        "Mean Precision": scores["test_precision"].mean(),
        "Mean Recall": scores["test_recall"].mean()
    })

cv_results_df = (
    pd.DataFrame(cv_results)
    .sort_values("Mean F1", ascending=False)
    .reset_index(drop=True)
)

display(cv_results_df.round(4))

,Model,Mean F1,F1 Std,Mean Precision,Mean Recall
0,LogisticRegression_C10,0.7456,0.0074,0.7663,0.7262
1,LinearSVC_C1,0.7446,0.0057,0.7672,0.7235
2,LogisticRegression_C1,0.7373,0.0086,0.8020,0.6823
3,NaiveBayes_alpha1,0.7230,0.0033,0.8740,0.6166


### Cross-Validation Results and Model Selection

The following results were obtained in the Kaggle environment using
Scikit-learn 1.6.1 and three-fold GroupKFold cross-validation.

| Model | Mean F1 | F1 Std | Mean Precision | Mean Recall |
|---|---:|---:|---:|---:|
| Logistic Regression (C=10) | 0.7456 | 0.0074 | 0.7663 | 0.7262 |
| LinearSVC (C=1) | 0.7446 | 0.0057 | 0.7672 | 0.7235 |
| Logistic Regression (C=1) | 0.7373 | 0.0086 | 0.8020 | 0.6823 |
| MultinomialNB (alpha=1) | 0.7230 | 0.0033 | 0.8740 | 0.6166 |

Logistic Regression with C=10 achieved the highest mean F1-score
in this execution, but its advantage over LinearSVC was approximately
0.001, which is too small to establish statistical superiority.

These results differ slightly from the original local experiment.
Differences in library versions or cross-validation fold assignments
are possible explanations, but the exact cause has not been established.

The project retains LinearSVC (C=1.0) as its previously selected
candidate to preserve consistency with the documented development
workflow and first official Kaggle submission.

The original development validation F1-score was 0.7588.
This is not an independent final performance estimate.

In [8]:
for fold_number, (train_indices, val_indices) in enumerate(folds, start=1):
    fold_train_texts = set(texts.iloc[train_indices])
    fold_val_texts = set(texts.iloc[val_indices])

    overlap = fold_train_texts.intersection(fold_val_texts)

    assert len(overlap) == 0

    print(
        f"Fold {fold_number}: "
        f"train={len(train_indices)}, "
        f"validation={len(val_indices)}, "
        f"text_overlap={len(overlap)}"
    )

Fold 1: train=4055, validation=2028, text_overlap=0
Fold 2: train=4055, validation=2028, text_overlap=0
Fold 3: train=4056, validation=2027, text_overlap=0


## Final Model Training and Submission

After completing the development experiments, the selected
TF-IDF + LinearSVC Pipeline is refitted using all 7,613 labeled tweets.

This final Pipeline is intended for inference on the official Kaggle
test dataset, which contains 3,263 unlabeled tweets.

Because the previous validation observations are now included in
training, their earlier F1-score must not be reused as an independent
evaluation of this refitted model.

In [9]:
final_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

final_pipeline.fit(
    train_df["text"],
    train_df["target"]
)

print("Final model trained successfully.")
print("Training observations:", len(train_df))
print("Classifier:", final_pipeline.named_steps["classifier"])

Final model trained successfully.
Training observations: 7613
Classifier: LinearSVC(max_iter=5000, random_state=42)


In [10]:
test_predictions = final_pipeline.predict(test_df["text"])

submission = pd.DataFrame({
    "id": test_df["id"].to_numpy(),
    "target": test_predictions
})

assert submission.shape == (3263, 2)
assert submission.columns.tolist() == ["id", "target"]
assert submission["id"].equals(test_df["id"])
assert submission["id"].equals(sample_submission["id"])
assert submission["id"].is_unique
assert not submission.isna().any().any()
assert set(submission["target"]).issubset({0, 1})

output_path = Path("/kaggle/working/submission.csv")
submission.to_csv(output_path, index=False)

print("Submission saved:", output_path)
print("Shape:", submission.shape)
print("\nPredicted classes:")
print(submission["target"].value_counts().sort_index())

display(submission.head())

Submission saved: /kaggle/working/submission.csv
Shape: (3263, 2)

Predicted classes:
target
0    1982
1    1281
Name: count, dtype: int64


,id,target
0,0,1
1,2,1
2,3,1
3,9,1
4,11,1


In [11]:
error_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LinearSVC(
        C=1.0,
        max_iter=5000,
        random_state=42
    ))
])

error_pipeline.fit(dev_train["text"], dev_train["target"])
validation_predictions = error_pipeline.predict(dev_val["text"])

error_df = dev_val[["id", "text", "target"]].copy()
error_df["prediction"] = validation_predictions

false_positives = error_df[
    (error_df["target"] == 0) &
    (error_df["prediction"] == 1)
]

false_negatives = error_df[
    (error_df["target"] == 1) &
    (error_df["prediction"] == 0)
]

print("False positives:", len(false_positives))
display(false_positives.head(3))

print("False negatives:", len(false_negatives))
display(false_negatives.head(3))

False positives: 132


,id,text,target,prediction
26,38,Was in NYC last week!,0,1
35,53,On plus side LOOK AT THE SKY LAST NIGHT IT WAS...,0,1
80,117,mom: 'we didn't get home as fast as we wished'...,0,1


False negatives: 182


,id,text,target,prediction
4,7,Just got sent this photo from Ruby #Alaska as ...,1,0
7,13,I'm on top of the hill and I can see a fire in...,1,0
51,74,Man wife get six years jail for setting ablaze...,1,0


## Conclusions and Limitations

This notebook reproduced a complete text classification workflow using
TF-IDF and traditional machine learning algorithms.

### Key Findings

- Group-aware splitting prevented identical texts from crossing
  training and validation partitions.
- LinearSVC achieved an F1-score of 0.7588 on the development
  validation split.
- Three-fold cross-validation produced closely competing results
  for LinearSVC and Logistic Regression.
- The final LinearSVC Pipeline was trained on all 7,613 labeled tweets.
- A validated submission file containing 3,263 predictions was generated.

### Previous Competition Result

The project's first official Kaggle submission achieved an
F1-score of **0.79313**.

This score was obtained from a previous submission generated by the
GitHub project, not from a new submission of this notebook.

### Limitations

The dataset contains repeated texts, conflicting labels, and
potentially ambiguous language.

Grouping identical texts does not prevent all forms of leakage,
including near-duplicate or event-related overlap.

The development validation set was also used during model comparisons
and should not be treated as a completely independent final test set.

The model is an educational classification prototype, not a
production-ready emergency detection system.

### Project Repository

[Disaster Tweet Classifier on GitHub](https://github.com/Guilherme5G/Disaster-tweet-classifier)